# Hyperparameter Tuning — Expert

## Dataset Validation
Run this cell to verify that your datasets are present and correctly formatted.

In [ ]:
# --- DATASET VALIDATION ---
import os
import pandas as pd

def validate_dataset(filepath, expected_columns=None, avoid_columns=None):
    if not os.path.exists(filepath):
        print(f'❌ ERROR: Dataset not found at {filepath}')
        return False
    try:
        df = pd.read_csv(filepath, nrows=5)
        print(f'✅ SUCCESS: Dataset found at {filepath} (Columns: {df.shape[1]})')
        if expected_columns:
            missing = [c for c in expected_columns if c not in df.columns]
            if missing:
                print(f'⚠️ WARNING: Missing expected columns: {missing}')
                return False
        if avoid_columns:
            forbidden = [c for c in avoid_columns if c in df.columns]
            if forbidden:
                print(f'❌ ERROR: Found forbidden columns {forbidden}. Wrong dataset!')
                return False
        return True
    except Exception as e:
        print(f'❌ ERROR: Could not read dataset: {e}')
        return False

print('Validating Santander Dataset...')
validate_dataset('../data/train.csv', expected_columns=['target', 'var_0'], avoid_columns=['Survived', 'PassengerId'])

In [1]:

from pathlib import Path
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUT_TABLES = PROJECT_DIR / "output" / "tables"
OUT_FIGURES = PROJECT_DIR / "output" / "figures"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42


In [2]:

def load_bank_data():
    path = DATA_DIR / "bank.csv"
    df = pd.read_csv(path)
    target_col = "deposit" if "deposit" in df.columns else "y"
    X = df.drop(columns=[target_col])
    y = df[target_col].map({"yes": 1, "no": 0}) if df[target_col].dtype == "object" else df[target_col]
    categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    numeric_cols = X.select_dtypes(exclude=["object", "category"]).columns.tolist()
    preprocess = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
            ("num", "passthrough", numeric_cols),
        ]
    )
    return df, X, y, preprocess

df, X, y, preprocess = load_bank_data()
df.shape, y.value_counts().to_dict()


KeyError: "['y'] not found in axis"

In [3]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint
import pandas as pd
import time

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

param_distributions = {
    "clf__n_estimators": randint(50, 301),
    "clf__max_depth": [3, 5, 8, 12, None],
    "clf__min_samples_split": randint(2, 11),
    "clf__min_samples_leaf": randint(1, 6),
    "clf__criterion": ["gini", "entropy"],
}

start = time.time()

halving = HalvingRandomSearchCV(
    estimator=pipe,
    param_distributions=param_distributions,
    factor=2,
    resource="n_samples",
    max_resources="auto",
    min_resources="smallest",
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)

pipe = Pipeline([
    ("preprocess", preprocess),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

halving.fit(X, y)

halving_runtime = time.time() - start

expert_summary = pd.DataFrame([{
    "method": "halving_random_search",
    "best_score": halving.best_score_,
    "runtime_seconds": halving_runtime,
    "best_params": str(halving.best_params_)
}])

expert_summary

NameError: name 'preprocess' is not defined

In [4]:

halving_results = pd.DataFrame(halving.cv_results_)
keep_cols = [
    "iter", "n_resources", "mean_test_score", "std_test_score", "rank_test_score", "params"
]
halving_results[keep_cols].to_csv(OUT_TABLES / "07_expert_halving_all_trials.csv", index=False)

plot_df = halving_results.groupby("iter", as_index=False)["mean_test_score"].max()
plt.figure(figsize=(7, 4))
plt.plot(plot_df["iter"], plot_df["mean_test_score"], marker="o")
plt.xlabel("Halving iteration")
plt.ylabel("Best CV accuracy in iteration")
plt.title("Successive Halving progress")
plt.tight_layout()
plt.savefig(OUT_FIGURES / "05_expert_halving_progress.png", dpi=160)
plt.show()


AttributeError: 'HalvingRandomSearchCV' object has no attribute 'cv_results_'

In [5]:

all_summaries = []
for file_name in [
    "01_beginner_grid_random_results.csv",
    "04_advanced_bayes_random_results.csv",
    "06_expert_halving_summary.csv",
]:
    path = OUT_TABLES / file_name
    if path.exists():
        all_summaries.append(pd.read_csv(path))

if all_summaries:
    final_summary = pd.concat(all_summaries, ignore_index=True)
    final_summary.to_csv(OUT_TABLES / "08_all_levels_final_summary.csv", index=False)
    display(final_summary)


,level,method,dataset,target,best_score,runtime_seconds,best_params
0,beginner,GridSearchCV,bank.csv,y,0.895156,8.739833,"{'clf__max_depth': None, 'clf__n_estimators': ..."
1,beginner,RandomizedSearchCV,bank.csv,y,0.895156,1.988535,"{'clf__n_estimators': 200, 'clf__max_depth': N..."
